In [45]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.impute import KNNImputer
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings

warnings.filterwarnings('ignore')

In [46]:
# df = pd.read_csv('./used_cars_price_predictor_project/data/pakwheels_data.csv')
df = sns.load_dataset('titanic')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [47]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [48]:
df.drop('deck', axis=1, inplace=True)
df.drop('alive', axis=1, inplace=True)

df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

df['age'] = KNNImputer().fit_transform(df[['age']])

In [49]:
X = df.drop('fare', axis=1)
y = df['fare']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [50]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categoric_features = X.select_dtypes(include=['category', 'object']).columns

In [51]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categoric_features)
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categoric_features),
    ],
    remainder='passthrough'
)

In [52]:
models = {

    'LinearRegression': (
        Pipeline([
            ('preprocessor', preprocessor),
            ('model', LinearRegression())
        ]),
        {}
    ),

    'DecisionTreeRegressor': (
        Pipeline([
            ('preprocessor', preprocessor_tree),
            ('model', DecisionTreeRegressor())
        ]),
        {
            'model__max_depth': [None, 5, 10],
            'model__splitter': ['best', 'random']
        }
    ),

    'RandomForestRegressor': (
        Pipeline([
            ('preprocessor', preprocessor_tree),
            ('model', RandomForestRegressor())
        ]),
        {
            'model__n_estimators': [10, 100, 1000],
            'model__max_depth': [None, 5, 10]
        }
    ),

    'KNeighborsRegressor': (
        Pipeline([
            ('preprocessor', preprocessor),
            ('model', KNeighborsRegressor())
        ]),
        {
            'model__n_neighbors': np.arange(3, 100, 2),
            'model__weights': ['uniform', 'distance']
        }
    ),

    'XGBRegressor': (
        Pipeline([
            ('preprocessor', preprocessor_tree),
            ('model', XGBRegressor())
        ]),
        {
            'model__n_estimators': [10, 100, 1000],
            'model__learning_rate': [0.1, 0.01, 0.001]
        }
    ),

    'SVR': (
        Pipeline([
            ('preprocessor', preprocessor),
            ('model', SVR())
        ]),
        {
            'model__kernel': ['rbf', 'poly', 'sigmoid'],
            'model__C': [0.1, 1, 10],
            'model__gamma': [1, 0.1, 0.01],
            'model__epsilon': [0.1, 0.01, 0.001]
        }
    ),
}

In [53]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

name = 'XGBRegressor'
scores = cross_val_score(models[name][0], X_train, y_train, cv=kfold, scoring='r2')
mean_r2 = np.mean(scores)

print(f"{name}: {mean_r2:.4f}")

XGBRegressor: 0.2714


In [54]:
results = {
    "LR": [],
    "DTR": [],
    "RFR": [],
    "KNR": [],
    "XGBR": [],
    "SVR": [],
}

In [55]:
for (name, (model, params)), acronym in zip(models.items(), results):
    grid = GridSearchCV(model, params, cv=5, n_jobs=-1)
    
    grid.fit(X_train, y_train)
    
    best_grid = grid.best_estimator_
    y_pred = best_grid.predict(X_test)
    
    print(f"------------------ {name} ------------------")
    print(r2_score(y_pred, y_test))
    print(mean_absolute_error(y_pred, y_test))
    print(mean_absolute_percentage_error(y_pred, y_test))
    print(f'Best Params: {grid.best_params_}')
    print(f'Best Estimator: {grid.best_estimator_}')
    print('\n\n')
    results[acronym].append(r2_score(y_pred, y_test))
    results[acronym].append(mean_absolute_error(y_pred, y_test))
    results[acronym].append(mean_absolute_percentage_error(y_pred, y_test))

------------------ LinearRegression ------------------
0.10457038454905898
17.719379251312876
1.143559733534803
Best Params: {}
Best Estimator: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['survived', 'pclass', 'age', 'sibsp', 'parch'], dtype='object')),
                                                 ('cat', OneHotEncoder(),
                                                  Index(['sex', 'embarked', 'class', 'who', 'embark_town'], dtype='object'))])),
                ('model', LinearRegression())])



------------------ DecisionTreeRegressor ------------------
0.4837174096816357
13.037626259007965
0.32093728949309086
Best Params: {'model__max_depth': 5, 'model__splitter': 'best'}
Best Estimator: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat', OneHotEncoder(),
      

In [56]:
result_df = pd.DataFrame(results, index=['R2', 'MAE', 'MAPE'])
result_df

,LR,DTR,RFR,KNR,XGBR,SVR
R2,0.104570,0.483717,0.476609,0.180672,-0.318777,0.495869
MAE,17.719379,13.037626,13.076677,14.968328,18.276363,10.862313
MAPE,1.143560,0.320937,0.317510,0.357182,0.504740,0.331428


In [57]:
df_transformed = result_df.T

filtered_models = df_transformed[
    (df_transformed['R2'] > 0) &
    (df_transformed['MAE'] < 15) &
    (df_transformed['MAPE'] < 0.40)
]

sorted_models = filtered_models.sort_values(
    by=['R2', 'MAE', 'MAPE'],
    ascending=[False, True, True]
)

print("---- Candidate Models ----")
print(filtered_models, end='\n\n')

print("---- Best Model ----")
print(sorted_models.index[0])

---- Candidate Models ----
           R2        MAE      MAPE
DTR  0.483717  13.037626  0.320937
RFR  0.476609  13.076677  0.317510
KNR  0.180672  14.968328  0.357182
SVR  0.495869  10.862313  0.331428

---- Best Model ----
SVR
